# DataMate trên Colab Pro: baseline, so model, thu dữ liệu, fine-tune, nộp bài

**Quy chế mục 4.3 cấm chia sẻ data cuộc thi cho bên thứ ba:** KHÔNG đưa data lên GitHub. Mã ở GitHub (private), data ở Google Drive cá nhân.
Chạy lần lượt từng ô. Chưa ô nào được chạy thử trên Colab thật; ô nào lỗi thì gửi lại thông báo.

Runtime: chọn GPU ở *Runtime > Change runtime type* (A100 là tốt nhất cho 14B; L4 chạy 14B bản AWQ; T4 chỉ 7B).


In [ ]:
# 1. Kiểm runtime
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())


In [ ]:
# 2. Drive + mã nguồn. Token GitHub lưu ở Colab Secrets (biểu tượng chìa khóa), tên GITHUB_TOKEN. KHÔNG dán token vào notebook.
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["GH"] = userdata.get("GITHUB_TOKEN")
REPO = "<tai-khoan>/<ten-repo>"            # <-- sửa
if not os.path.isdir("/content/datamate_agent"):
    !git clone https://$GH@github.com/$REPO.git /content/datamate_agent
%cd /content/datamate_agent
!git pull


In [ ]:
# 3. Cài thư viện. vLLM nặng và có thể đổi bản torch: chỉ bật khi dùng A100/L4, rồi RESTART runtime và chạy lại từ ô 2.
INSTALL_VLLM = False
!pip -q install pytest openpyxl accelerate bitsandbytes peft huggingface_hub transformers
if INSTALL_VLLM:
    !pip -q install vllm


In [ ]:
# 4. Cấu hình đường dẫn (data nằm TRONG Drive cá nhân, không nằm trong repo)
D = "/content/drive/MyDrive"
DATA = f"{D}/datamate_data"                 # gồm train/ và public_test/
OUT = f"{D}/datamate_out"                   # cache, log, tóm tắt, submission: ở Drive nên bị ngắt session vẫn còn
TRAIN, TEST = f"{DATA}/train", f"{DATA}/public_test"
COMMON = f"--train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local --auto-model"
!python -m pytest -q


## 5. Smoke test: 3 câu, đo giây/câu và VRAM (chạy `!nvidia-smi` ở ô khác trong lúc chạy)

In [ ]:
!python -m evaluation.run_eval {COMMON} --max-biz 1 --per-biz 3 --run-name smoke

## 6. Baseline trên HOLDOUT (6 business giữ lại): so các model bằng PA, không bằng danh tiếng
Mỗi model chạy bằng một tiến trình riêng để giải phóng VRAM. Cache riêng theo `--run-name`.
Chênh lệch nhỏ hơn sai số (in ở ô 7) thì chưa kết luận được: tăng số câu holdout hoặc dùng thêm business.

In [ ]:
MODELS = [
    ("qwen25coder14b", "--local-model Qwen/Qwen2.5-Coder-14B-Instruct --local-engine vllm"),
    ("qwen3_14b",      "--local-model Qwen/Qwen3-14B --local-engine vllm --no-thinking"),
    ("qwen25coder7b",  "--local-model Qwen/Qwen2.5-Coder-7B-Instruct --local-engine hf"),   # mốc so; cho T4 dùng dòng này
]
for name, flags in MODELS:
    !python -m evaluation.run_eval --train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local {flags} --subset holdout --holdout-n 6 --per-biz 15 --run-name {name}


In [ ]:
# 7. So sánh (kèm cảnh báo khi chênh lệch nằm trong sai số)
!python -m evaluation.compare --out-dir {OUT}


## 8. Các cờ nên A/B trên holdout (đổi MỘT thứ mỗi lần, cùng model)
`--no-helpers`, `--no-hints`, `--rulebook-select`, `--n-samples 5 --agree-min 3`, `--exemplars <kho>`. Giữ cái nào làm PA tăng vượt sai số.

In [ ]:
BEST = "--local-model Qwen/Qwen2.5-Coder-14B-Instruct --local-engine vllm"      # <-- thay bằng model thắng ở ô 7
for name, flags in [("best_nohelpers", "--no-helpers"), ("best_rulebooksel", "--rulebook-select"), ("best_n5", "--n-samples 5 --agree-min 3")]:
    !python -m evaluation.run_eval --train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local {BEST} {flags} --subset holdout --holdout-n 6 --per-biz 15 --run-name {name}
!python -m evaluation.compare --out-dir {OUT}


## 9. Thu dữ liệu SFT từ chính model (nhóm FIT, KHÔNG chạm holdout)
Sinh đủ N mẫu cho mỗi câu, giữ mọi mẫu cho ra đúng đáp án chuẩn (sạch, đúng ngay lần đầu). Chỉ dùng tập train.
Có thể chạy làm nhiều lần (tăng `--max-biz`/`--per-biz`) vì cache nối tiếp được.

In [ ]:
!python -m evaluation.run_eval {COMMON} {BEST} --subset fit --holdout-n 6 --per-biz 40 --n-samples 4 --agree-min 2 --collect-all --export-sft --sft-all --build-exemplars {OUT}/bank.jsonl --run-name collect1
!wc -l {OUT}/sft_train.jsonl


## 10. Fine-tune QLoRA (chạy `--dry-run` trước để xem bao nhiêu mẫu vừa độ dài)
Base phải là model GỐC fp16. A100 40GB: thử 14B với `--max-len 8192`; nếu hết VRAM thì dùng 7B hoặc giảm `--max-len`.

In [ ]:
BASE = "Qwen/Qwen2.5-Coder-14B-Instruct"
!python -m training.finetune_qlora --data {OUT}/sft_train.jsonl --base {BASE} --out {OUT}/lora --max-len 8192 --dry-run


In [ ]:
!python -m training.finetune_qlora --data {OUT}/sft_train.jsonl --base {BASE} --out {OUT}/lora --max-len 8192
!python -m training.finetune_qlora --merge --base {BASE} --out {OUT}/lora --merged {OUT}/merged


In [ ]:
# 11. Đo model đã fine-tune trên CÙNG holdout rồi so với bản gốc
!python -m evaluation.run_eval --train-dir {TRAIN} --test-dir {TEST} --out-dir {OUT} --backend local --local-model {OUT}/merged --local-engine vllm --subset holdout --holdout-n 6 --per-biz 15 --run-name finetuned
!python -m evaluation.compare --out-dir {OUT}


## 12. Dự đoán public test (1000 câu) -> submission.jsonl đủ id, có kiểm định. Nộp tối đa 5 lượt/ngày.

In [ ]:
FINAL = "--local-model Qwen/Qwen2.5-Coder-14B-Instruct --local-engine vllm"       # hoặc {OUT}/merged nếu bản fine-tune thắng
!python -m app.main --test-dir {TEST} --out-dir {OUT} --backend local {FINAL} --run-name final
